In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from ariel_pred.config import CalibrationConfig
from ariel_pred.dataset import DataLoaderAndCalibrator, LabelsLoader
from ariel_pred.features import SergeiOldFeaturesExtractor
from ariel_pred.models import SegeiOldCNNTrainer, SergeiOldInference
from ariel_pred.preprocessing import SergeiDataSmoother
from ariel_pred.transit import WindowBasedPhaseDetector

In [4]:
input_data_folder: str = "../data/raw_subset"
output_data_folder: str = "../data/processed/sergei"
output_model_folder: str = "../models/sergei"
submission_file: str = "./submission.csv"
stop_at_calibration: bool = False
stop_at_feature_extraction: bool = False
stop_at_model_training: bool = False

In [5]:
# Path setup
input_data_path = Path(input_data_folder)
output_data_path = Path(output_data_folder)
output_model_path = Path(output_model_folder)

if not input_data_path.exists():
    raise ValueError(f"Input data folder {input_data_path} does not exist")

os.makedirs(output_data_path, exist_ok=True)
os.makedirs(output_model_path, exist_ok=True)

calibrated_train_data_file = output_data_path / "calibrated_train_data.npy"
calibrated_test_data_file = output_data_path / "calibrated_test_data.npy"
train_features_file = output_data_path / "train_features.npy"
test_features_file = output_data_path / "test_features.npy"
train_labels_file = output_data_path / "train_labels.npy"

In [6]:
# Calibration and preprocessing
calibration_config = CalibrationConfig(
    data_path=input_data_path,
    binning=1,
    airs_lower_channel=0,
    airs_upper_channel=356,
    preprocessing_n_jobs=4,
)
signal_processor = DataLoaderAndCalibrator(cfg=calibration_config)
data_smoother = SergeiDataSmoother(window_size=3)
if not calibrated_train_data_file.exists():
    print("Calibrating and saving train data...")
    train_data = signal_processor.process_all_data("train")
    train_data = np.array([data_smoother.smooth(signal) for signal in train_data])
    np.save(calibrated_train_data_file, train_data)
else:
    print("Loading calibrated train data...")
    train_data = np.load(calibrated_train_data_file, allow_pickle=True)

test_data = signal_processor.process_all_data("test")
test_data = np.array([data_smoother.smooth(signal) for signal in test_data])
np.save(calibrated_test_data_file, test_data)

Loading calibrated train data...


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# Feature Extraction
transit_detector = WindowBasedPhaseDetector()
feature_extractor = SergeiOldFeaturesExtractor(phase_detector=transit_detector)
if not train_features_file.exists():
    print("Extracting and saving train features...")
    train_features = feature_extractor.extract_features(train_data)
    np.save(train_features_file, train_features)
else:
    print("Loading train features...")
    train_features = np.load(train_features_file, allow_pickle=True)
    
print("Extracting and saving test features...")
test_features = feature_extractor.extract_features(test_data)
np.save(test_features_file, test_features)

Loading train features...
Extracting and saving test features...


  0%|          | 0/1 [00:00<?, ?it/s]/Users/ibrahimhabib/Desktop/development/ariel-2025/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ibrahimhabib/Desktop/development/ariel-2025/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ibrahimhabib/Desktop/development/ariel-2025/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ibrahimhabib/Desktop/development/ariel-2025/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ibrahimhabib/Desktop/development/ariel-2025/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: Runti

In [8]:
# Labels loading
if not train_labels_file.exists():
    print("Loading and saving train labels...")
    labels_loader = LabelsLoader(base_data_path=str(input_data_path))
    train_labels = labels_loader.load_labels()
    np.save(train_labels_file, train_labels)
else:
    print("Loading train labels...")
    train_labels = np.load(train_labels_file, allow_pickle=True)

Loading train labels...


In [9]:
# Model Training
cnn_train_data = train_features.transpose(0, 2, 1)
cnn_test_data = test_features.transpose(0, 2, 1)
if len(os.listdir(output_model_path)) == 0:
    print("Training model...")
    model_trainer = SegeiOldCNNTrainer(device=torch.device("mps"))
    model_trainer.train(cnn_train_data, train_labels, output_model_path)

Training model...


Training SergeiOldCNN:   0%|          | 0/1000 [00:00<?, ?it/s]

Fold 1/5


Training SergeiOldCNN:  21%|██        | 206/1000 [00:05<00:20, 38.10it/s]

fold 1 train 1e-06 valid 1e-05 rmse 0.013815 ariel -1.2713281824000865e+25
Fold 2/5


...

fold 3 train 3e-06 valid 1e-06 rmse 0.015144 ariel -1.7172926927311839e+25
Fold 4/5


Training SergeiOldCNN:  80%|████████  | 801/1000 [00:19<00:06, 32.96it/s]

fold 4 train 3e-06 valid 0.0 rmse 0.008068 ariel -6435602.672433
Fold 5/5


Training SergeiOldCNN: 100%|██████████| 1000/1000 [00:24<00:00, 40.29it/s]

fold 5 train 3e-06 valid 1e-06 rmse 0.015171 ariel -1.7389875682073882e+25


In [10]:
# Inference
print("Running inference on test data...")
inference_model = SergeiOldInference(models_dir=output_model_path, device=torch.device("mps"))
predictions = inference_model.predict(cnn_test_data)

Running inference on test data...


In [17]:
sample_submission = pd.read_csv(input_data_path / "sample_submission.csv")
sample_submission.iloc[:, 1:] = predictions

In [18]:
sample_submission

,planet_id,wl_1,wl_2,wl_3,wl_4,wl_5,wl_6,wl_7,wl_8,wl_9,...,sigma_274,sigma_275,sigma_276,sigma_277,sigma_278,sigma_279,sigma_280,sigma_281,sigma_282,sigma_283
0,1103775,0.002508,0.00251,0.002511,0.002513,0.002514,0.002514,0.002515,0.002516,0.002516,...,5.458733e-07,5.458733e-07,5.458733e-07,5.458733e-07,5.458733e-07,5.458733e-07,5.458733e-07,5.458733e-07,5.458733e-07,5.458733e-07
